# Overfit One SST-2 Activation Oracle Sample

This notebook loads one cached SST-2 `TrainingDataPoint`, attaches a LoRA adapter to the base model, and repeatedly trains on that one sample. It is meant as a tiny end-to-end sanity check for activation materialization, injection, forward pass, and loss.

In [2]:
from pathlib import Path
import os
import random
import sys

import torch
from peft import LoraConfig, PeftModel, get_peft_model
from torch.nn.utils import clip_grad_norm_
from transformers import BitsAndBytesConfig

THIS_DIR = Path.cwd()
# WORKSPACE_ROOT = Path('/Users/gloria/dev/ai_safety/evil_oracles')
WORKSPACE_ROOT = THIS_DIR.parent
REPO_DIR = WORKSPACE_ROOT / 'activation_oracles'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from nl_probes.configs.sft_config import SelfInterpTrainingConfig
from nl_probes.dataset_classes.act_dataset_manager import DatasetLoaderConfig
from nl_probes.dataset_classes.classification import ClassificationDatasetConfig, ClassificationDatasetLoader
from nl_probes.sft import train_features_batch
from nl_probes.utils.activation_utils import get_hf_submodule, get_text_only_lora_targets
from nl_probes.utils.common import load_model, load_tokenizer, set_seed
from nl_probes.utils.dataset_utils import construct_batch, materialize_missing_steering_vectors


ModuleNotFoundError: No module named 'slist'

In [ ]:
model_name = 'Qwen/Qwen3-8B'
dataset_folder = 'sft_training_data'
output_dir = WORKSPACE_ROOT / 'ao_minimal_training' / 'checkpoints_single_sample_overfit'

layer_percents = [25, 50, 75]
hook_layer = 1
save_acts = False
num_train = 6000
seed = 42
lr = 1e-4
num_steps = 100
load_in_8bit = True

set_seed(seed)
device = torch.device('cuda:0')
dtype = torch.bfloat16


In [ ]:
params = ClassificationDatasetConfig(
    classification_dataset_name='sst2',
    max_window_size=1,
    min_window_size=1,
    min_end_offset=-1,
    max_end_offset=-5,
    num_qa_per_sample=2,
)
dataset_cfg = DatasetLoaderConfig(
    custom_dataset_params=params,
    num_train=num_train,
    num_test=0,
    splits=['train'],
    model_name=model_name,
    layer_percents=layer_percents,
    save_acts=save_acts,
    batch_size=1,
    dataset_folder=str(dataset_folder),
)
loader = ClassificationDatasetLoader(dataset_config=dataset_cfg, model_kwargs={})
cache_path = Path(loader.dataset_config.dataset_folder) / loader.get_dataset_filename('train')
print(cache_path)
if not cache_path.exists():
    raise FileNotFoundError(f'Missing cached dataset: {cache_path}')

training_data = loader.load_dataset('train')
dp = training_data[0]
print(f'Loaded {len(training_data)} datapoints; using one sample')
print('target_output:', dp.target_output)
print('layer:', dp.layer)
print('num activation positions:', len(dp.positions))


In [ ]:
cfg = SelfInterpTrainingConfig(
    model_name=model_name,
    hook_onto_layer=hook_layer,
    layer_percents=layer_percents,
    train_batch_size=1,
    lr=lr,
    steering_coefficient=1.0,
    save_dir=str(output_dir),
    seed=seed,
).finalize(dataset_loaders=[loader])

tokenizer = load_tokenizer(model_name)
model_kwargs = {'device_map': {'': 'cuda:0'}}
if load_in_8bit:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=dtype)

model = load_model(model_name, dtype, **model_kwargs)
model.enable_input_require_grads()

target_modules = cfg.lora_target_modules
vlm_targets = get_text_only_lora_targets(cfg.model_name)
if vlm_targets and target_modules == 'all-linear':
    target_modules = vlm_targets
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=0.0,
    target_modules=target_modules,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=True)
model.print_trainable_parameters()
model.train()
submodule = get_hf_submodule(model, hook_layer, use_lora=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)


In [ ]:
losses = []
optimizer.zero_grad()
for step in range(num_steps):
    batch_points = materialize_missing_steering_vectors([dp], tokenizer, model)
    batch = construct_batch(batch_points, tokenizer, device)
    loss = train_features_batch(cfg, batch, model, submodule, device, dtype)
    loss.backward()
    clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
    optimizer.step()
    optimizer.zero_grad()
    losses.append(float(loss.item()))
    print(f'step {step:03d} loss {losses[-1]:.6f}')


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Saved overfit LoRA adapter to {output_dir}')


## Optional generation check

This checks whether the overfit adapter can generate the target answer from the same activation-injected prompt. It is a qualitative check; the loss curve above is the main sanity signal.

In [ ]:
from nl_probes.utils.eval import run_evaluation

model.eval()
result = run_evaluation(
    eval_data=[dp],
    model=model,
    tokenizer=tokenizer,
    submodule=submodule,
    device=device,
    dtype=dtype,
    global_step=0,
    lora_path=None,
    eval_batch_size=1,
    steering_coefficient=cfg.steering_coefficient,
    generation_kwargs={'do_sample': False, 'max_new_tokens': 5},
)
print('target:', dp.target_output)
print('generated:', result[0].api_response)
